In [ ]:
!pip install xlrd

# 0 - Imports

In [ ]:
# Fundamental libraries
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Main model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import ParameterGrid
from collections import defaultdict

# Models extension
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Uncertainty analysis
from sklearn.utils import resample

from sklearn.metrics import confusion_matrix

# 1 - Data collection

## 1.1 - Extract and download data to folder

In [ ]:
urls = [
    'http://tennis-data.co.uk/2000/2000.xls', 
    #'http://tennis-data.co.uk/2001/2001.xls', 
    'http://tennis-data.co.uk/2002/2002.xls',
    'http://tennis-data.co.uk/2003/2003.xls', 
    'http://tennis-data.co.uk/2004/2004.xls',
    'http://tennis-data.co.uk/2005/2005.xls', 
    'http://tennis-data.co.uk/2006/2006.xls',
    'http://tennis-data.co.uk/2007/2007.xls', 
    #'http://tennis-data.co.uk/2008/2008.zip', 
    'http://tennis-data.co.uk/2009/2009.xls', 
    'http://tennis-data.co.uk/2010/2010.xls',
    'http://tennis-data.co.uk/2011/2011.xls', 
    'http://tennis-data.co.uk/2012/2012.xls',
    'http://tennis-data.co.uk/2013/2013.xlsx', 
    'http://tennis-data.co.uk/2014/2014.xlsx',
    'http://tennis-data.co.uk/2015/2015.xlsx', 
    'http://tennis-data.co.uk/2016/2016.xlsx', 
    'http://tennis-data.co.uk/2017/2017.xlsx', 
    'http://tennis-data.co.uk/2018/2018.xlsx',
    'http://tennis-data.co.uk/2019/2019.xlsx', 
    'http://tennis-data.co.uk/2020/2020.xlsx',
    'http://tennis-data.co.uk/2021/2021.xlsx', 
    'http://tennis-data.co.uk/2022/2022.xlsx',
    'http://tennis-data.co.uk/2023/2023.xlsx',
    'http://tennis-data.co.uk/2024/2024.xlsx',
    'http://tennis-data.co.uk/2025/2025.xlsx',
    'http://tennis-data.co.uk/2026/2026.xlsx'
]

# Folder to save files
save_folder = "datasets"
os.makedirs(save_folder, exist_ok=True)

for url in urls:
    filename = url.split("/")[-1]
    filepath = os.path.join(save_folder, filename)
    
    print(f"Downloading {filename}...")
    
    response = requests.get(url)
    with open(filepath, "wb") as f:
        f.write(response.content)

## 1.2 - Read data and insert to dataframe

In [ ]:
folder_path = "datasets"

# Dictionary to save dataframes
dfs = {}

for file in os.listdir(folder_path):
    filepath = os.path.join(folder_path, file)
    if file.endswith(".xlsx"):
        df = pd.read_excel(filepath)
        
    elif file.endswith(".xls"):
        df = pd.read_excel(filepath, engine="xlrd")
        
    else:
        continue
    
    # Use filename as key
    key = os.path.splitext(file)[0]
    dfs[key] = df
    
    print(f"Loaded {file} → key: '{key}', shape: {df.shape}")

print("\nTotal datasets loaded:", len(dfs))

## 1.3 - Data cleaning and inspection

In [ ]:
# Get sets of column names for each year/df
column_sets = [set(df.columns) for df in dfs.values()]

# Find columns present in all dfs
common_columns = set.intersection(*column_sets)

print(f"Columns present in all datasets ({len(common_columns)}):")
print(common_columns)

In [ ]:
for year, df in dfs.items():
    unique_cols = set(df.columns) - common_columns
    if unique_cols:
        print(f"Columns only in {year}: {unique_cols}")

### 1.3.1 - Drop 2000-2004, 2026 dataframes

In [ ]:
years_to_drop = ['2000','2001', '2002', '2003', '2004', '2026']

for year in years_to_drop:
    if year in dfs:
        del dfs[year]
    else:
        print(f"{year} not found in dfs")

### 1.3.2 - Drop bookmaker columns and keep used columns

In [ ]:
# List of columns to keep
columns_to_keep = ['WPts', 'LPts', 'L5', 'Location', 'Winner', 'Date', 'ATP', 'Comment', 'W2', 'W5', 'L1', 'L3', 'Lsets', 'W4', 'Loser', 'Wsets', 'W1', 'LRank', 'WRank', 'Round', 'Surface', 'Best of', 'L2', 'W3', 'Tournament', 'Series', 'L4', 'Court']  

# Drop bookmaker columns across all DataFrames
for key, df in dfs.items():
    # Keep only the columns that exist in both the DataFrame and kept-columns list
    keep_cols = [col for col in df.columns if col in columns_to_keep]
    dfs[key] = df[keep_cols].copy()  # assign back a clean copy
    print(f"{key}: kept columns {dfs[key].columns.tolist()}")

### 1.3.3 - Rows and columns inspection

In [ ]:
total_rows = 0
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    # number of rows
    n_rows = int(df.shape[0])
    total_rows += n_rows
    # number of columns
    n_cols = df.shape[1]
    print(f"Rows: {n_rows}, Columns: {n_cols}")
print("Total rows:",total_rows)

### 1.3.4 - Value checking

In [ ]:
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    
    # Total number of rows
    n_rows = df.shape[0]
    n_cols = df.shape[1]
    print(f"Rows: {n_rows}, Columns: {n_cols}")
    
    # Check missing / null values per column
    missing = df.isnull().sum()
    print("Missing values per column:")
    print(missing)

### 1.3.5 - Identify categorical and numerical columns, and unique categorical values

In [ ]:
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    
    # Separate column types
    numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"Numerical columns: {numerical_cols}")
    # for col in numerical_cols:
    #     unique_vals = df[col].unique()
    #     print(f"  {col}: {len(unique_vals)} unique values -> {unique_vals[:10]}{'...' if len(unique_vals) > 10 else ''}")
    
    print(f"\nCategorical columns: {categorical_cols}")
    for col in categorical_cols:
        unique_vals = df[col].unique()
        print(f"  {col}: {len(unique_vals)} unique values -> {unique_vals[:10]}{'...' if len(unique_vals) > 10 else ''}")

### 1.3.6 - Merge into 1 dataframe

In [ ]:
full_df = pd.concat(dfs.values(), ignore_index=True)

In [ ]:
full_df.isna().sum()

In [ ]:
# Additional checks and fill missing values
# 1. Ranks & Points
max_rank = full_df[['WRank', 'LRank']].max().max()
full_df['WRank'] = full_df['WRank'].fillna(max_rank + 1)
full_df['LRank'] = full_df['LRank'].fillna(max_rank + 1)
full_df['WPts'] = pd.to_numeric(full_df['WPts'], errors='coerce').fillna(0)
full_df['LPts'] = pd.to_numeric(full_df['LPts'], errors='coerce').fillna(0)

# 2. Set Scores & Format
full_df['Best of'] = full_df.apply(lambda r: 5 if r['Series'] == 'Grand Slam' else 3 if pd.isna(r['Best of']) else r['Best of'], axis=1)
full_df[['Wsets', 'Lsets']] = full_df[['Wsets', 'Lsets']].fillna(0)

# 3. Game columns (W1-L5)
# Filter out matches that weren't actually played (Walkovers, Retirements) to ensure game/set stats are valid for Block 1 (Fatigue) calculations later
full_df = full_df[full_df['Comment'] == 'Completed']
full_df = full_df[~((full_df['Comment'] == 'Completed') & (df['Wsets'].isna()))]

### 1.3.7 - Dataset statistics

In [ ]:
# Date range
print("Date range:", full_df["Date"].min(), "to", full_df["Date"].max())

In [ ]:
print(list(full_df.columns))

In [ ]:
# Ensure numeric
full_df["WRank"] = pd.to_numeric(full_df["WRank"], errors="coerce")
full_df["LRank"] = pd.to_numeric(full_df["LRank"], errors="coerce")
full_df["WPts"] = pd.to_numeric(full_df["WPts"], errors="coerce")
full_df["LPts"] = pd.to_numeric(full_df["LPts"], errors="coerce")

# Clean data
df_rank = full_df.dropna(subset=["WRank", "LRank"]).copy()
df_rank = df_rank[(df_rank["WRank"] <= 500) & (df_rank["LRank"] <= 500)]

df_points = full_df.dropna(subset=["WPts", "LPts"]).copy()
df_points = df_points[(df_points["WPts"] > 0) & (df_points["LPts"] > 0)]

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# Rank histogram
axes[0].hist([df_rank["WRank"], df_rank["LRank"]],
             bins=50,
             stacked=True,
             label=["Winner Rank", "Loser Rank"])
axes[0].set_title("Winner vs Loser Rank Distribution", fontsize=18)
axes[0].set_xlabel("ATP Rank", fontsize=14)
axes[0].set_ylabel("Match Count", fontsize=14)
axes[0].legend()
axes[0].grid(True)

# Points histogram
axes[1].hist([df_points["WPts"], df_points["LPts"]],
             bins=50,
             stacked=True,
             label=["Winner Points", "Loser Points"])
axes[1].set_yscale("log")
axes[1].set_title("Winner vs Loser Points Distribution", fontsize=18)
axes[1].set_xlabel("ATP Points", fontsize=14)
axes[1].set_ylabel("Match Count (Log Scale)", fontsize=14)
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("tennis_plot_histogram.png", dpi=300)
plt.show()

In [ ]:
# Surface counts
surface_counts = full_df["Surface"].value_counts()
print(surface_counts)
plt.figure(figsize=(6,4))
surface_counts.plot(kind='bar', color='green')
plt.title("Surface Type Distribution")
plt.xlabel("Surface")
plt.ylabel("Count")
plt.show()

In [ ]:
# Court type
indoor_counts = full_df["Court"].value_counts()
indoor_counts.plot(kind='bar', color="red")
plt.title("Matches by Court Type")
plt.xlabel("Type")
plt.ylabel("Count")
plt.show()

In [ ]:
# Unique player and tournaments
unique_players = set(full_df["Winner"]).union(set(full_df["Loser"]))
print("Unique players:", len(unique_players))
print("Unique tournaments:", full_df["Tournament"].nunique())

In [ ]:
# Best of counts
best_of_counts = full_df["Best of"].value_counts()
print(best_of_counts)
plt.figure(figsize=(6,4))
best_of_counts.plot(kind='bar', color='orange')
plt.title("Match Format (Best of)")
plt.xlabel("Best of")
plt.ylabel("Count")
plt.show()


# 2 - Modeling

## 2.1 - Additional filtering and cleaning

In [ ]:
df = full_df.copy()

# Convert Date to datetime and sort to preserve chronological order
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)


# --- OPTIMIZATION: ELO RATINGS ---
def calculate_elo(df, k_factor=32):
    elo_dict = {}
    winner_elos = []
    loser_elos = []
    
    for idx, row in df.iterrows():
        w = row['Winner']
        l = row['Loser']
        
        # Get current Elo or default to 1500 for new players
        w_elo = elo_dict.get(w, 1500)
        l_elo = elo_dict.get(l, 1500)
        
        winner_elos.append(w_elo)
        loser_elos.append(l_elo)
        
        # Calculate expected win probability based on current Elo
        expected_w = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
        expected_l = 1 / (1 + 10 ** ((w_elo - l_elo) / 400))
        
        # Update dictionary with new Elo after match result
        elo_dict[w] = w_elo + k_factor * (1 - expected_w)
        elo_dict[l] = l_elo + k_factor * (0 - expected_l)
        
    df['Winner_Elo'] = winner_elos
    df['Loser_Elo'] = loser_elos
    return df

# Apply Elo calculations sequentially across the entire history
df = calculate_elo(df)

# Convert ranks and points to numeric, coercing errors (like 'NR' for Not Ranked) to NaN
for col in ['WRank', 'LRank', 'WPts', 'LPts']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## 2.2 - Data structuring

In [ ]:
# Convert winner/loser to Player A / Player B
np.random.seed(42)
# Create a boolean mask to randomly swap Winner/Loser 50% of the time
swap = np.random.rand(len(df)) > 0.5

# Assign players
df['Player_A'] = np.where(swap, df['Loser'], df['Winner'])
df['Player_B'] = np.where(swap, df['Winner'], df['Loser'])

# Target variable: 1 if Player A wins, 0 if Player A loses
df['Target_A_Win'] = np.where(swap, 0, 1)

# Map Ranks and Points based on the swap
df['Rank_A'] = np.where(swap, df['LRank'], df['WRank'])
df['Rank_B'] = np.where(swap, df['WRank'], df['LRank'])
df['Pts_A'] = np.where(swap, df['LPts'], df['WPts'])
df['Pts_B'] = np.where(swap, df['WPts'], df['LPts'])

## 2.3 - Block 0

### 0.1 - Feature engineering

#### 0.1.1 - Rank and points

In [ ]:
# Point and rank differences from the perspective of Player A
df['Rank_Diff_A_B'] = df['Rank_A'] - df['Rank_B']
df['Pts_Diff_A_B'] = df['Pts_A'] - df['Pts_B']

#### 0.1.2 - ELO

In [ ]:
# --- ELO DIFFERENTIAL ---
def calculate_elo(df, k_factor=32):
    elo_dict = {}          # global ELO: {player: rating}
    surface_elo_dict = {}  # surface ELO: {player: {surface: rating}}

    winner_elos, loser_elos = [], []
    winner_surface_elos, loser_surface_elos = [], []

    for idx, row in df.iterrows():
        w, l, surface = row['Winner'], row['Loser'], row['Surface']

        # --- Global ELO (unchanged) ---
        w_elo = elo_dict.get(w, 1500)
        l_elo = elo_dict.get(l, 1500)

        winner_elos.append(w_elo)
        loser_elos.append(l_elo)

        exp_w = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
        exp_l = 1 / (1 + 10 ** ((w_elo - l_elo) / 400))

        elo_dict[w] = w_elo + k_factor * (1 - exp_w)
        elo_dict[l] = l_elo + k_factor * (0 - exp_l)

        # --- Surface-specific ELO (new) ---
        w_srf = surface_elo_dict.setdefault(w, {}).get(surface, 1500)
        l_srf = surface_elo_dict.setdefault(l, {}).get(surface, 1500)

        winner_surface_elos.append(w_srf)
        loser_surface_elos.append(l_srf)

        exp_w_srf = 1 / (1 + 10 ** ((l_srf - w_srf) / 400))
        exp_l_srf = 1 / (1 + 10 ** ((w_srf - l_srf) / 400))

        surface_elo_dict[w][surface] = w_srf + k_factor * (1 - exp_w_srf)
        surface_elo_dict[l][surface] = l_srf + k_factor * (0 - exp_l_srf)

    df['Winner_Elo']         = winner_elos
    df['Loser_Elo']          = loser_elos
    df['Winner_Surface_Elo'] = winner_surface_elos
    df['Loser_Surface_Elo']  = loser_surface_elos
    return df

df = calculate_elo(df)

# --- ELO MAPPING ---
df['Elo_A'] = np.where(swap, df['Loser_Elo'], df['Winner_Elo'])
df['Elo_B'] = np.where(swap, df['Winner_Elo'], df['Loser_Elo'])
# --- ELO DIFFERENTIAL ---
df['Elo_Diff_A_B'] = df['Elo_A'] - df['Elo_B']

# --- SURFACE ELO MAPPING ---
df['Surface_Elo_A'] = np.where(swap, df['Loser_Surface_Elo'], df['Winner_Surface_Elo'])
df['Surface_Elo_B'] = np.where(swap, df['Winner_Surface_Elo'], df['Loser_Surface_Elo'])
# --- SURFACE ELO DIFFERENTIAL ---
df['Surface_Elo_Diff_A_B'] = df['Surface_Elo_A'] - df['Surface_Elo_B']

#### 0.1.3 - H2H

In [ ]:
h2h_dict = defaultdict(lambda: defaultdict(int))  # h2h_dict[winner][loser] = prior wins

h2h_wins_A_list = []
h2h_wins_B_list = []

for _, row in df.iterrows():
    pA, pB = row['Player_A'], row['Player_B']

    # Read PRIOR wins before this match (no leakage)
    wins_A = h2h_dict[pA][pB]
    wins_B = h2h_dict[pB][pA]

    h2h_wins_A_list.append(wins_A)
    h2h_wins_B_list.append(wins_B)

    # Update AFTER recording pre-match state
    if row['Target_A_Win'] == 1:
        h2h_dict[pA][pB] += 1
    else:
        h2h_dict[pB][pA] += 1

df['H2H_Wins_A'] = h2h_wins_A_list
df['H2H_Wins_B'] = h2h_wins_B_list
df['H2H_Total']  = df['H2H_Wins_A'] + df['H2H_Wins_B']

# Raw differential (main signal: positive means A has beaten B more often)
df['H2H_Wins_Diff'] = df['H2H_Wins_A'] - df['H2H_Wins_B']

# Bayesian-smoothed win rate (handles first meetings gracefully; alpha=3 = mild prior toward 0.5)
alpha = 3
df['H2H_Win_Rate_A'] = (df['H2H_Wins_A'] + alpha) / (df['H2H_Total'] + 2 * alpha)

### 0.2 - Create final Block 0 differential features

In [ ]:
# Create a clean dataframe for the baseline model
baseline_cols = ['Date', 'Tournament', 'Surface', 'Player_A', 'Player_B', 
                 'Rank_Diff_A_B', 'Pts_Diff_A_B', 'Elo_Diff_A_B',
                 'H2H_Wins_Diff', 'H2H_Win_Rate_A', 'H2H_Total',
                 'Target_A_Win', 'Surface_Elo_Diff_A_B']
model_df = df[baseline_cols].copy()

### 0.3 - Dataset split

In [ ]:
# 1. Training Set (2005 to 2019)
train_set = model_df[model_df['Date'].dt.year <= 2019].copy()

# 2. Validation Set (2020 to 2023)
val_set = model_df[(model_df['Date'].dt.year >= 2020) & (model_df['Date'].dt.year <= 2023)].copy()

# 3. Testing Set (2024 to 2025)
test_set = model_df[model_df['Date'].dt.year >= 2024].copy()

# Verify the splits
print(f"Total Matches: {len(model_df)}")
print(f"Training Set: {len(train_set)} matches ({len(train_set)/len(model_df):.1%})")
print(f"Validation Set: {len(val_set)} matches ({len(val_set)/len(model_df):.1%})")
print(f"Testing Set: {len(test_set)} matches ({len(test_set)/len(model_df):.1%})")

## 2.4 - Block 1

### 1.1 - Calculate Total Games and Sets for each historical match

In [ ]:
# Use original 'df' from before swapping A/B to calculate match stats
game_cols = ['W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5']
for col in game_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Sum across the row for total games
df['Total_Games'] = df[game_cols].sum(axis=1)

# Sum total sets
df['Total_Sets'] = pd.to_numeric(df['Wsets'], errors='coerce').fillna(0) + \
                   pd.to_numeric(df['Lsets'], errors='coerce').fillna(0)

### 1.2 - Create player history dataframe

In [ ]:
winners = df[['Date', 'Winner', 'Total_Sets', 'Total_Games']].rename(columns={'Winner': 'Player'})
losers = df[['Date', 'Loser', 'Total_Sets', 'Total_Games']].rename(columns={'Loser': 'Player'})

# Combine, sort chronologically, and drop duplicates if a player played twice in one day
player_history = pd.concat([winners, losers]).sort_values(by=['Player', 'Date']).drop_duplicates(subset=['Player', 'Date'])

### 1.3 - Calculate rest days since last match

In [ ]:
player_history['Days_Since_Last_Match'] = player_history.groupby('Player')['Date'].diff().dt.days

### 1.4 - Calculate 14-day rolling load

In [ ]:
# --- NEW CODE: EWMA FATIGUE (Replaces old rolling window) ---
# Sort chronologically by player
player_history = player_history.sort_values(by=['Player', 'Date'])

# Calculate Exponential Moving Average (span=14 matches)
# .shift(1) ensures the EWMA only includes matches prior to the current row
ewma_metrics = player_history.groupby('Player')[['Total_Sets', 'Total_Games']].apply(
    lambda x: x.shift(1).ewm(span=14, adjust=False).mean()
).reset_index(level=0, drop=True)

player_history['Total_Sets_EWMA'] = ewma_metrics['Total_Sets']
player_history['Total_Games_EWMA'] = ewma_metrics['Total_Games']

# Fill NA for a player's very first match
player_history.fillna({'Total_Sets_EWMA': 0, 'Total_Games_EWMA': 0}, inplace=True)

# Create the final metrics dataframe
fatigue_metrics = player_history[['Date', 'Player', 'Days_Since_Last_Match', 'Total_Sets_EWMA', 'Total_Games_EWMA']].copy()

### 1.5 - Merge Block 1 features into model_df

In [ ]:
# Merge for Player A
model_df = pd.merge(model_df, fatigue_metrics.add_suffix('_A'), 
                    left_on=['Date', 'Player_A'], right_on=['Date_A', 'Player_A'], how='left').drop(columns=['Date_A'])

# Merge for Player B
model_df = pd.merge(model_df, fatigue_metrics.add_suffix('_B'), 
                    left_on=['Date', 'Player_B'], right_on=['Date_B', 'Player_B'], how='left').drop(columns=['Date_B'])

### 1.6 - Create final Block 1 differential features

In [ ]:
# --- NEW CODE: EWMA DIFFERENTIALS ---
model_df['Sets_Played_EWMA_Diff'] = model_df['Total_Sets_EWMA_A'] - model_df['Total_Sets_EWMA_B']
model_df['Games_Played_EWMA_Diff'] = model_df['Total_Games_EWMA_A'] - model_df['Total_Games_EWMA_B']
model_df['Rest_Days_Diff'] = model_df['Days_Since_Last_Match_A'] - model_df['Days_Since_Last_Match_B']

## 2.5 - Block 2

### 2.1 - Create a long DataFrame for Surface Performance

In [ ]:
winners_sfc = df[['Date', 'Winner', 'Surface']].copy()
winners_sfc['Player'] = winners_sfc['Winner']
winners_sfc['Won'] = 1
winners_sfc = winners_sfc.drop(columns=['Winner'])

losers_sfc = df[['Date', 'Loser', 'Surface']].copy()
losers_sfc['Player'] = losers_sfc['Loser']
losers_sfc['Won'] = 0
losers_sfc = losers_sfc.drop(columns=['Loser'])

# Combine and sort chronologically
surface_history = pd.concat([winners_sfc, losers_sfc]).sort_values(by=['Player', 'Surface', 'Date'])

### 2.2 - Calculate historical wins and matches (excluding current match)

In [ ]:
grouped = surface_history.groupby(['Player', 'Surface'])

# cumcount() starts at 0, which perfectly represents the number of matches played BEFORE the current one
surface_history['Historical_Matches_On_Surface'] = grouped.cumcount()

# Cumulative sum of wins minus the current match's result gives previous wins
surface_history['Historical_Wins_On_Surface'] = grouped['Won'].cumsum() - surface_history['Won']

### 2.3 - Calculate win rate on surface

In [ ]:
# If a player has never played on this surface before, default their win rate to 0
surface_history['Surface_Win_Rate'] = np.where(
    surface_history['Historical_Matches_On_Surface'] > 0, 
    surface_history['Historical_Wins_On_Surface'] / surface_history['Historical_Matches_On_Surface'], 
    0 
)

### 2.4 - Clean up for merging

In [ ]:
surface_features = surface_history[['Date', 'Player', 'Surface', 'Surface_Win_Rate', 'Historical_Matches_On_Surface']]
# Drop duplicates in the rare case a player plays two matches on the same surface in one day
surface_features = surface_features.drop_duplicates(subset=['Date', 'Player', 'Surface'])

### 2.5 - Merge block 2 features to model_df

In [ ]:
# Merge for Player A
model_df = pd.merge(model_df, 
                    surface_features.rename(columns={'Player': 'Player_A', 
                                                     'Surface_Win_Rate': 'Surface_Win_Rate_A',
                                                     'Historical_Matches_On_Surface': 'Matches_On_Surface_A'}), 
                    on=['Date', 'Player_A', 'Surface'], 
                    how='left')

# Merge for Player B
model_df = pd.merge(model_df, 
                    surface_features.rename(columns={'Player': 'Player_B', 
                                                     'Surface_Win_Rate': 'Surface_Win_Rate_B',
                                                     'Historical_Matches_On_Surface': 'Matches_On_Surface_B'}), 
                    on=['Date', 'Player_B', 'Surface'], 
                    how='left')

# Fill any lingering NaNs (e.g., players completely missing from the history merge) with 0
model_df.fillna({'Surface_Win_Rate_A': 0, 'Surface_Win_Rate_B': 0}, inplace=True)

### 2.6 - Create final Block 2 differential features

In [ ]:
model_df['Surface_Win_Rate_Diff'] = model_df['Surface_Win_Rate_A'] - model_df['Surface_Win_Rate_B']

## 2.6 - Block 3

In [58]:
# Map 'Series' to an ordinal Importance Tier
series_map = {
    'Grand Slam': 5, 'Masters Cup': 5, 'ATP Tour Finals': 5,
    'Masters': 4, 'Masters 1000': 4,
    'International Gold': 3, 'ATP500': 3,
    'International': 2, 'ATP250': 2
}

# Map 'Round' to an ordinal Depth metric (Momentum)
round_map = {
    'The Final': 7, 'Final': 7,
    'Semifinals': 6,
    'Quarterfinals': 5,
    '4th Round': 4,
    '3rd Round': 3,
    '2nd Round': 2,
    '1st Round': 1,
    'Round Robin': 1
}

# Extract directly from the original 'df' and map to model_df
model_df['Tournament_Tier'] = df['Series'].map(series_map).fillna(1).astype(int)
model_df['Tournament_Depth'] = df['Round'].map(round_map).fillna(1).astype(int)

# 3 - Model training and validation evaluation

## 3.1 - Feature blocks

In [59]:
# Define Feature Blocks based on engineered columns
block_0 = ['Rank_Diff_A_B', 'Pts_Diff_A_B', 'Elo_Diff_A_B', 'H2H_Wins_Diff', 'H2H_Win_Rate_A', 'H2H_Total']
block_1 = ['Sets_Played_EWMA_Diff', 'Games_Played_EWMA_Diff', 'Rest_Days_Diff']
block_2 = ['Surface_Win_Rate_Diff', 'Surface_Elo_Diff_A_B']
block_3 = ['Tournament_Tier', 'Tournament_Depth']

# Define the rigorous incremental model configurations
feature_sets = {
    '1. Baseline (Block 0)': block_0,
    '1.1 Baseline (Block 1)': block_1,
    '1.2 Baseline (Block 2)': block_2,
    '1.3 Baseline (Block 3)': block_3,
    '2. Baseline + Fatigue (Block 0, 1)': block_0 + block_1,
    '3. Baseline + Surface (Block 0, 2)': block_0 + block_2,
    '4. Baseline + Context (Block 0, 3)': block_0 + block_3,
    '5. Baseline + Fatig/Surf (Block 0, 1, 2)': block_0 + block_1 + block_2,
    '6. Baseline + Fatig/Cont (Block 0, 1, 3)': block_0 + block_1 + block_3,
    '7. Baseline + Surf/Cont (Block 0, 2, 3)': block_0 + block_2 + block_3,
    '8. All Features (Block 0, 1, 2, 3)': block_0 + block_1 + block_2 + block_3
}

## 3.2 - Data prep for modeling

In [60]:
# --- THE FIX: Re-split the dataset now that model_df has all new columns ---
train_set = model_df[model_df['Date'].dt.year <= 2019].copy()
val_set = model_df[(model_df['Date'].dt.year >= 2020) & (model_df['Date'].dt.year <= 2023)].copy()
test_set = model_df[model_df['Date'].dt.year >= 2024].copy()

# Extract all features used
all_features = list(set(block_0 + block_1 + block_2))

# Clean and prepare training/validation sets
train_clean = train_set.dropna(subset=all_features + ['Target_A_Win']).copy()
val_clean = val_set.dropna(subset=all_features + ['Target_A_Win']).copy()

y_train = train_clean['Target_A_Win']
y_val = val_clean['Target_A_Win']

print(f"Training on {len(train_clean)} matches, Validating on {len(val_clean)} matches.\n")

Training on 37747 matches, Validating on 8620 matches.



## 3.3 - Train and evaluate incremental models

In [61]:
results = []
C_values = [0.01, 0.1, 1.0, 10.0]

for name, features in feature_sets.items():
    X_train = train_clean[features]
    X_val = val_clean[features]

    best_loss_LR = float('inf')
    best_auc_LR = 0
    best_c = 1.0

    # Grid search over C on the validation set
    for C in C_values:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, random_state=42, max_iter=1000))
        model.fit(X_train, y_train)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        loss = log_loss(y_val, y_pred_proba)
        auc = roc_auc_score(y_val, y_pred_proba)

        if loss < best_loss_LR:
            best_loss_LR = loss
            best_auc_LR = auc
            best_c = C

    results.append({
        'Model configuration': name,
        'Log-Loss': round(best_loss_LR, 4),
        'AUC': round(best_auc_LR, 4),
        'Best C': best_c
    })
    print(f"{name} | Best C={best_c} | Log-Loss={best_loss_LR:.4f} | AUC={best_auc_LR:.4f}")

1. Baseline (Block 0) | Best C=0.01 | Log-Loss=0.6205 | AUC=0.7094
1.1 Baseline (Block 1) | Best C=0.01 | Log-Loss=0.6903 | AUC=0.5345
1.2 Baseline (Block 2) | Best C=0.01 | Log-Loss=0.6341 | AUC=0.6887
1.3 Baseline (Block 3) | Best C=0.01 | Log-Loss=0.6932 | AUC=0.5000
2. Baseline + Fatigue (Block 0, 1) | Best C=0.01 | Log-Loss=0.6201 | AUC=0.7097
3. Baseline + Surface (Block 0, 2) | Best C=0.01 | Log-Loss=0.6187 | AUC=0.7116
4. Baseline + Context (Block 0, 3) | Best C=0.01 | Log-Loss=0.6206 | AUC=0.7094
5. Baseline + Fatig/Surf (Block 0, 1, 2) | Best C=0.01 | Log-Loss=0.6182 | AUC=0.7119
6. Baseline + Fatig/Cont (Block 0, 1, 3) | Best C=0.01 | Log-Loss=0.6201 | AUC=0.7097
7. Baseline + Surf/Cont (Block 0, 2, 3) | Best C=0.01 | Log-Loss=0.6188 | AUC=0.7115
8. All Features (Block 0, 1, 2, 3) | Best C=0.01 | Log-Loss=0.6183 | AUC=0.7119


## 3.4 -  Results

In [62]:
results_df = pd.DataFrame(results).set_index('Model configuration')

# Sort by Log-Loss to clearly see the best performing model
print("--- Validation Set Evaluation Results ---")
print(results_df.sort_values('Log-Loss'))

--- Validation Set Evaluation Results ---
                                          Log-Loss     AUC  Best C
Model configuration                                               
5. Baseline + Fatig/Surf (Block 0, 1, 2)    0.6182  0.7119    0.01
8. All Features (Block 0, 1, 2, 3)          0.6183  0.7119    0.01
3. Baseline + Surface (Block 0, 2)          0.6187  0.7116    0.01
7. Baseline + Surf/Cont (Block 0, 2, 3)     0.6188  0.7115    0.01
2. Baseline + Fatigue (Block 0, 1)          0.6201  0.7097    0.01
6. Baseline + Fatig/Cont (Block 0, 1, 3)    0.6201  0.7097    0.01
1. Baseline (Block 0)                       0.6205  0.7094    0.01
4. Baseline + Context (Block 0, 3)          0.6206  0.7094    0.01
1.2 Baseline (Block 2)                      0.6341  0.6887    0.01
1.1 Baseline (Block 1)                      0.6903  0.5345    0.01
1.3 Baseline (Block 3)                      0.6932  0.5000    0.01


## 3.5 - Alternative algorithm comparison 1 - Random Forest

In [ ]:
# 1. Isolate the winning feature set
best_features = block_0 + block_1 + block_2

X_train_best = train_clean[best_features]
X_val_best = val_clean[best_features]

# 2. Initialize the Random Forest model
# Note: Tree-based models don't require StandardScaler, so we can fit directly
# We constrain max_depth to prevent overfitting on the training data
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

# 3. Train the new model
print("Training Random Forest...")
rf_model.fit(X_train_best, y_train)

# 4. Predict probabilities on the validation set
rf_pred_proba = rf_model.predict_proba(X_val_best)[:, 1]

# 5. Evaluate and Compare
rf_auc = roc_auc_score(y_val, rf_pred_proba)
rf_loss = log_loss(y_val, rf_pred_proba)

print(f"Random Forest Model:      Log-Loss = {rf_loss:.4f} | AUC = {rf_auc:.4f}")

# Optional: View Feature Importances to see what the Random Forest prioritized
importances = pd.Series(rf_model.feature_importances_, index=best_features).sort_values(ascending=False)
print("\n--- Random Forest Feature Importances ---")
print(importances)

## 3.6 - Alternative algorithm comparison 2 - XGBoost

In [ ]:
# hyperparameter tuning
xgb_param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [300, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

best_xgb_loss = float('inf')
best_xgb_auc = 0
best_xgb_params = {}

print("Tuning XGBoost...")
for params in ParameterGrid(xgb_param_grid):
    xgb = XGBClassifier(**params, random_state=42, eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
    xgb.fit(X_train_best, y_train)
    preds = xgb.predict_proba(X_val_best)[:, 1]
    loss = log_loss(y_val, preds)
    auc = roc_auc_score(y_val, preds)

    if loss < best_xgb_loss:
        best_xgb_loss = loss
        best_xgb_auc = auc
        best_xgb_params = params

print(f"\nBest XGBoost Params: {best_xgb_params}")
print(f"XGBoost Best:         Log-Loss = {best_xgb_loss:.4f} | AUC = {best_xgb_auc:.4f}")

# --- XGBoost Feature Importances ---
best_xgb_model = XGBClassifier(**best_xgb_params, random_state=42, eval_metric='logloss', use_label_encoder=False)
best_xgb_model.fit(X_train_best, y_train)
xgb_importances = pd.Series(best_xgb_model.feature_importances_, index=best_features).sort_values(ascending=False)
print("\n--- XGBoost Feature Importances ---")
print(xgb_importances)

## 3.7 - Alternative algorithm comparison 3 - LightGBM

In [ ]:
# hyperparameter tuning
lgbm_param_grid = {
    'num_leaves': [15, 31, 50],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [300, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

best_lgbm_loss = float('inf')
best_lgbm_auc = 0
best_lgbm_params = {}

print("Tuning LightGBM...")
for params in ParameterGrid(lgbm_param_grid):
    # verbose=-1 suppresses the excessive LightGBM warning outputs
    lgbm = LGBMClassifier(**params, random_state=42, verbose=-1, n_jobs=-1)
    lgbm.fit(X_train_best, y_train)
    preds = lgbm.predict_proba(X_val_best)[:, 1]
    loss = log_loss(y_val, preds)
    auc = roc_auc_score(y_val, preds)

    if loss < best_lgbm_loss:
        best_lgbm_loss = loss
        best_lgbm_auc = auc
        best_lgbm_params = params

print(f"Best LightGBM Params: {best_lgbm_params}")

# --- Feature Importances ---
best_lgbm_model = LGBMClassifier(**best_lgbm_params, random_state=42, verbose=-1)
best_lgbm_model.fit(X_train_best, y_train)
importances = pd.Series(best_lgbm_model.feature_importances_, index=best_features).sort_values(ascending=False)

# Normalize importances so they sum to 1.0 (to match XGBoost's default output style for easier reading)
importances = importances / importances.sum()

print(f"\n--- {best_lgbm_model.__class__.__name__} Feature Importances ---")
print(importances)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Re-train the optimal Logistic Regression model explicitly
# Using C=0.01 on the 'best_features' (All 3 blocks) as found in your tuning phase
best_lr_model = make_pipeline(
    StandardScaler(), 
    LogisticRegression(C=0.01, random_state=42, max_iter=1000)
)
best_lr_model.fit(X_train_best, y_train)

# 2. Update the dictionary with the correct variables
models_dict = {
    'Logistic Regression': best_lr_model,      
    'Random Forest': rf_model,             
    'XGBoost': best_xgb_model,                 
    'LightGBM': best_lgbm_model                 
}

plt.figure(figsize=(10, 8))

# 3. Loop through each model to calculate and plot its ROC curve
for name, m in models_dict.items():
    # Predict probabilities on the validation set
    # CRITICAL FIX: Use X_val_best so it has all 3 blocks of features
    y_pred_proba = m.predict_proba(X_val_best)[:, 1] 
    
    # Calculate False Positive Rate (fpr) and True Positive Rate (tpr)
    fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    # Plot the curve
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

# Plot the 50/50 random guess baseline
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guess')

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=14)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=14)
plt.title('ROC Curve Comparison - All Models', fontsize=18, fontweight='bold')
plt.legend(loc="lower right", fontsize=13)
plt.grid(alpha=0.3)
plt.savefig("ROC_curve.png", dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()

# 4 - Final evaluation on test set

## 4.1 - Combine train and validation set

In [ ]:
train_val_combined = pd.concat([train_clean, val_clean], axis=0)

X_train_final = train_val_combined[best_features]
y_train_final = train_val_combined['Target_A_Win']

# Drop NaNs in the Test Set to ensure clean evaluation
test_clean = test_set.dropna(subset=best_features + ['Target_A_Win']).copy()
X_test = test_clean[best_features]
y_test = test_clean['Target_A_Win']

## 4.2 - Retrain using best algorithm and predict

In [ ]:
# --- Summary ---
print("--- Algorithm Comparison (Validation Set) ---")
print(f"Logistic Regression: Log-Loss = {best_loss_LR:.4f} | AUC = {best_auc_LR:.4f}")
print(f"Random Forest:       Log-Loss = {rf_loss:.4f} | AUC = {rf_auc:.4f}")
print(f"XGBoost:             Log-Loss = {best_xgb_loss:.4f} | AUC = {best_xgb_auc:.4f}")
print(f"LightGBM:            Log-Loss = {best_lgbm_loss:.4f} | AUC = {best_lgbm_auc:.4f}")

best_algorithm = ""

if best_loss_LR < rf_loss and best_loss_LR < best_xgb_loss and best_loss_LR < best_lgbm_loss: # use logistic regression
    best_algorithm = "Logistic Regression"
    print("\n[Using Logistic Regression as final model]")
    final_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_c, random_state=42, max_iter=1000))
    final_model.fit(X_train_final, y_train_final)
elif rf_loss < best_loss_LR and rf_loss < best_xgb_loss and rf_loss < best_lgbm_loss: # use random forest
    best_algorithm = "Random Forest"
    print("\n[Using Random Forest as final model]")
    final_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    final_model.fit(X_train_final, y_train_final)
elif best_xgb_loss < best_loss_LR and best_xgb_loss < rf_loss and best_xgb_loss < best_lgbm_loss: # use XGBoost
    best_algorithm = "XGBoost"
    print("\n[Using XGBoost as final model]")
    final_model = XGBClassifier(**best_xgb_params, random_state=42, eval_metric='logloss', use_label_encoder=False)
    final_model.fit(X_train_final, y_train_final)
elif best_lgbm_loss < best_loss_LR and best_lgbm_loss < rf_loss and best_lgbm_loss < best_xgb_loss: # use lightGBM
    best_algorithm = "LightGBM"
    print("\n[Using LightGBM as final model]")
    final_model = LGBMClassifier(**best_lgbm_params, random_state=42, verbose=-1)
    final_model.fit(X_train_final, y_train_final)

print(f"\nTraining Final Model on {len(X_train_final)} historical matches (2005-2023)...")
print(f"Evaluating on {len(X_test)} recent matches (2024-2025)...\n")

# Predict
y_test_pred_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = final_model.predict(X_test)

## 4.3 - Calculate final metrics and report

In [ ]:
test_auc = roc_auc_score(y_test, y_test_pred_proba)
test_loss = log_loss(y_test, y_test_pred_proba)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"=== FINAL TEST SET PERFORMANCE (2024-2025) - {best_algorithm} ===")
print(f"Log-Loss: {test_loss:.4f}")
print(f"AUC:      {test_auc:.4f}")
print(f"Accuracy: {test_accuracy:.2%}\n")

print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Player A Loses', 'Player A Wins']))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix:")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: Loss", "Actual: Win"],
    columns=["Pred: Loss", "Pred: Win"]
)

print(cm_df)

# 5 - Predicting match outcome

In [ ]:
def predict_match(player_A, player_B, surface, match_date, raw_df, model, best_features):
    match_date = pd.to_datetime(match_date)
    
    # Filter historical data to matches strictly BEFORE the prediction date
    history = raw_df[(pd.to_datetime(raw_df['Date']) < match_date) & 
                     (raw_df['Comment'] == 'Completed')].copy()

    def get_player_stats(player):
        # Filter for matches involving this player
        p_history = history[(history['Winner'] == player) | (history['Loser'] == player)].sort_values('Date')
        
        if p_history.empty:
            raise ValueError(f"No historical data found for {player}")
            
        last_match = p_history.iloc[-1]
        is_winner = last_match['Winner'] == player
        
        # 1. Rank & Points
        rank = last_match['WRank'] if is_winner else last_match['LRank']
        pts = last_match['WPts'] if is_winner else last_match['LPts']
        
        # 2. Elo (The model expects 'Winner_Elo' / 'Loser_Elo' columns from your calc_elo step)
        elo = last_match['Winner_Elo'] if is_winner else last_match['Loser_Elo']
        
        # 3. EWMA Fatigue (Fetching the values calculated in Block 1)
        # Note: If these aren't in your 'df', ensure you've merged your 'fatigue_metrics' back into 'df'
        sets_ewma = last_match.get('Total_Sets_EWMA', 0)
        games_ewma = last_match.get('Total_Games_EWMA', 0)
        
        # 4. Surface Win Rate
        s_history = p_history[p_history['Surface'] == surface]
        s_win_rate = len(s_history[s_history['Winner'] == player]) / len(s_history) if not s_history.empty else 0.5
        
        # 5. Surface Elo (If your model uses it)
        s_elo = last_match.get('Surface_Elo_A', 1500) if is_winner else last_match.get('Surface_Elo_B', 1500)

        return float(rank), float(pts), float(elo), float(sets_ewma), float(games_ewma), float(s_win_rate), float(s_elo)

    def get_h2h_stats(pA, pB):
        # Look for past matches between these two specifically
        h2h = history[((history['Winner'] == pA) & (history['Loser'] == pB)) | 
                      ((history['Winner'] == pB) & (history['Loser'] == pA))]
        
        if h2h.empty:
            return 0, 0, 0.5 # Default for new matchups
        
        wins_A = len(h2h[h2h['Winner'] == pA])
        total = len(h2h)
        return wins_A - (total - wins_A), total, wins_A / total

    # Extract Stats
    rA, ptA, eloA, sewmaA, gewmaA, swrA, seloA = get_player_stats(player_A)
    rB, ptB, eloB, sewmaB, gewmaB, swrB, seloB = get_player_stats(player_B)
    
    h2h_diff, h2h_total, h2h_rate = get_h2h_stats(player_A, player_B)

    # Construct feature dictionary to match your 'best_features' exactly
    features = {
        'Rank_Diff_A_B': rA - rB,
        'Pts_Diff_A_B': ptA - ptB,
        'Elo_Diff_A_B': eloA - eloB,
        'Sets_Played_EWMA_Diff': sewmaA - sewmaB,
        'Games_Played_EWMA_Diff': gewmaA - gewmaB,
        'Surface_Win_Rate_Diff': swrA - swrB,
        'Surface_Elo_Diff_A_B': seloA - seloB,
        'H2H_Wins_Diff': h2h_diff,
        'H2H_Total': h2h_total,
        'H2H_Win_Rate_A': h2h_rate,
        'Rest_Days_Diff': 0 # Optional: add rest days logic if required
    }

    # Create the inference row
    X_inference = pd.DataFrame([features])
    
    # Ensure all required columns are present (fill missing with 0 if necessary)
    for col in best_features:
        if col not in X_inference.columns:
            X_inference[col] = 0
            
    X_inference = X_inference[best_features]

    # Predict
    prob = model.predict_proba(X_inference)[0][1]
    
    print(f"--- PREDICTION: {player_A} vs {player_B} ---")
    print(f"Elo Diff: {eloA - eloB:+.1f} | Surface WR Diff: {swrA - swrB:+.1%}")
    print(f"Win Probability {player_A}: {prob:.1%}")
    print("-" * 40)

# Call with your best_features list
predict_match("Tien L.", "Medvedev D.", "Hard", "2026-04-15", df, best_xgb_model, best_features)

# 6 - Sensitivity analysis

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

plt.rcParams.update({
    'font.size': 14,          # base font size
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14
})

# Choose the top features you want to analyze
# Replace these with the exact column names from your 'best_features' list
features_to_analyze = ['Elo_Diff_A_B', 'Rank_Diff_A_B', 'Surface_Elo_Diff_A_B', 'Pts_Diff_A_B']

fig, ax = plt.subplots(2, 2, figsize=(12, 10))

# Plot Partial Dependence (Global Sensitivity)
# We use X_train_final to see how the model learned the feature landscapes
display = PartialDependenceDisplay.from_estimator(
    estimator=best_xgb_model,           # Replace with your trained XGBoost variable
    X=X_train_final, 
    features=features_to_analyze,
    feature_names=best_features,   # Ensures the axes are labeled correctly
    kind="both",                   # 'both' plots the average (PDP) and individual lines (ICE)
    subsample=100,                 # Sample 100 lines for visual clarity
    n_jobs=-1,
    ax=ax,
    line_kw={"color": "red", "linewidth": 2},       # PDP line style
    ice_lines_kw={"color": "blue", "alpha": 0.05}   # ICE lines style
)

plt.suptitle("Global Sensitivity Analysis (PDP & ICE)\nHow Features Affect Player A's Win Probability", 
             fontsize=18, fontweight='bold')
plt.subplots_adjust(top=0.85)
plt.savefig("sensitivity_analysis_plots.png", dpi=300, bbox_inches='tight')
plt.show()